In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROCESSED_DIR = Path('data/processed')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

calls = pd.read_parquet(PROCESSED_DIR / 'calls_trainval.parquet') 
with open(PROCESSED_DIR / 'split.json') as f:
    split_info = json.load(f)

print(calls.shape)
print(calls['split'].value_counts())
print(split_info)
calls.head()

In [ ]:
calls = calls.dropna(subset=['c_bid', 'c_ask'])

calls = calls[calls['c_bid'] > 0]

calls = calls[calls['c_ask'] > calls['c_bid']]

calls = calls[calls['dte'] > 1]

calls = calls.drop_duplicates(subset=['quote_date', 'expire_date', 'strike'])

intrinsic = np.maximum(calls['underlying_last'] - calls['strike'], 0)
calls = calls[calls['mid'] >= intrinsic]



In [ ]:
print(calls.shape)
calls[['mid', 'moneyness', 'dte']].describe()

In [ ]:
per_month = calls.groupby(calls['quote_date'].dt.to_period('M')).size()
per_month.plot(kind='bar', figsize=(10, 3.5), color= 'blue')
plt.title('Call quotes per month')
plt.ylabel('Rows')
plt.tight_layout(); plt.savefig(FIG_DIR / 'quotes_per_month.png')
plt.show()

In [ ]:
calls['moneyness'].clip(0.5, 1.5).hist(bins=100, figsize=(7, 3.5), color= 'blue')
plt.title('Moneyness S/K')
plt.xlabel('S/K  (>1 = in the money)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'moneyness_hist.png'); plt.show()

In [ ]:
calls['dte'].clip(upper=730).hist(bins=100, figsize=(7, 3.5), color= 'blue')
plt.title('Days to expiry')
plt.xlabel('DTE')
plt.tight_layout(); plt.savefig(FIG_DIR / 'dte_hist.png'); plt.show()

In [ ]:
spx = calls.groupby('quote_date')['underlying_last'].first()
spx.plot(figsize=(10, 3.5), color= 'blue')
plt.title('SPX close')
plt.ylabel('Index level')
plt.tight_layout(); plt.savefig(FIG_DIR / 'spx_level.png'); plt.show()

In [ ]:
day = calls['quote_date'].median().normalize()
one_day = calls[(calls['quote_date'] == calls.loc[calls['quote_date'] >= day, 'quote_date'].min())
                & calls['dte'].between(25, 35)]
plt.figure(figsize=(7, 3.5))
plt.scatter(one_day['moneyness'], one_day['c_iv'], s=8, color= 'blue')
plt.xlabel('S/K'); plt.ylabel('Implied vol')
plt.title(f'IV smile, ~30-day calls on {one_day["quote_date"].iloc[0].date()}')
plt.tight_layout(); plt.savefig(FIG_DIR / 'iv_smile.png'); plt.show()

In [ ]:
calls.to_parquet(PROCESSED_DIR / 'calls_clean_trainval.parquet', index=False)